# Nipoppy brain age demo

Federated SGD regression of age from FreeSurfer IDPs (DKT cortical thickness, aseg subcortical volumes, eTIV) and sex, on Nipoppy datasets.

## Training plan

In [ ]:
from typing import Callable, Optional

import numpy as np
import pandas as pd
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.dataset import NipoppyDataset
from fedbiomed.common.training_plans import FedSGDRegressor


class BrainAgeTrainingPlan(FedSGDRegressor):
    """Federated SGD regressor of age from FreeSurfer IDPs on Nipoppy studies."""

    # Neurobagel terms used in Nipoppy harmonized phenotypic TSV file
    # column names
    AGE = "nb:Age"
    SEX = "nb:Sex"
    DIAGNOSIS = "nb:Diagnosis"

    # values
    MALE = "snomed:248153007"
    FEMALE = "snomed:248152002"
    HEALTHY_CONTROL = "ncit:C94342"

    # encoding for sex variable
    SEX_ENCODING = {MALE: 0.0, FEMALE: 1.0}

    # FreeSurfer 7 cortical thickness, Desikan-Killiany-Tourville atlas
    DKT_REGIONS = [
        "caudalanteriorcingulate",
        "caudalmiddlefrontal",
        "cuneus",
        "entorhinal",
        "fusiform",
        "inferiorparietal",
        "inferiortemporal",
        "isthmuscingulate",
        "lateraloccipital",
        "lateralorbitofrontal",
        "lingual",
        "medialorbitofrontal",
        "middletemporal",
        "parahippocampal",
        "paracentral",
        "parsopercularis",
        "parsorbitalis",
        "parstriangularis",
        "pericalcarine",
        "postcentral",
        "posteriorcingulate",
        "precentral",
        "precuneus",
        "rostralanteriorcingulate",
        "rostralmiddlefrontal",
        "superiorfrontal",
        "superiorparietal",
        "superiortemporal",
        "supramarginal",
        "transversetemporal",
        "insula",
    ]
    THICKNESS_COLUMNS = [f"lh_{region}_thickness" for region in DKT_REGIONS] + [
        f"rh_{region}_thickness" for region in DKT_REGIONS
    ]

    # FreeSurfer 7 subcortical volumes (aseg) and intracranial volume
    SUBCORTICAL_STRUCTURES = [
        "Lateral-Ventricle",
        "Thalamus",
        "Caudate",
        "Putamen",
        "Pallidum",
        "Hippocampus",
        "Amygdala",
        "Accumbens-area",
    ]
    VOLUME_COLUMNS = (
        ["EstimatedTotalIntraCranialVol"]
        + [f"Left-{structure}" for structure in SUBCORTICAL_STRUCTURES]
        + [f"Right-{structure}" for structure in SUBCORTICAL_STRUCTURES]
    )

    # model inputs
    FEATURE_COLUMNS = [SEX] + THICKNESS_COLUMNS + VOLUME_COLUMNS
    N_FEATURES = len(FEATURE_COLUMNS)  # 80

    def init_dependencies(self):
        return [
            "from typing import Callable, Optional",
            "import numpy as np",
            "import pandas as pd",
            "from fedbiomed.common.datamanager import DataManager",
            "from fedbiomed.common.dataset import NipoppyDataset",
        ]

    @staticmethod
    def select_freesurfer_derivatives(dataset: NipoppyDataset) -> None:
        """Derivatives initializer: use the FreeSurfer/fs_stats versions installed on the node.

        Called by `NipoppyDataset.load` on the node, after `get_metadata()` is available
        and before the data is read.
        """
        pipelines = dataset.get_metadata()["installed_pipelines"]
        fs_version = pipelines["processing"]["freesurfer"][0]
        fs_stats_version = pipelines["extraction"]["fs_stats"][0]
        dataset.derivatives = [
            (
                "freesurfer",
                fs_version,
                f"idp/fs_stats-{fs_stats_version}/fs{fs_version}-{suffix}",
            )
            for suffix in ("aparc.DKTatlas-thickness.tsv", "aseg-volume.tsv")
        ]

    @classmethod
    def prepare_dataframe(
        cls, df: pd.DataFrame, controls_only: bool = True
    ) -> pd.DataFrame:
        """Whole-dataframe transform: select rows, encode and order columns.

        Runs on the node after session filtering. Returns the features in
        `FEATURE_COLUMNS` order (raw units) followed by the target.
        """
        if controls_only:
            df = df.loc[df[cls.DIAGNOSIS] == cls.HEALTHY_CONTROL]

        out = df[cls.FEATURE_COLUMNS + [cls.AGE]].copy()
        out[cls.SEX] = out[cls.SEX].map(cls.SEX_ENCODING)

        # drop rows with missing values (in the columns used only)
        out = out.dropna()
        if out.empty:
            raise ValueError("No participant left (check session and controls_only)")
        return out

    @classmethod
    def standardizer(cls, feature_stats: dict) -> Callable[[np.ndarray], np.ndarray]:
        """Sample-level transform: z-score one sample's features.

        `feature_stats` is `{"mean": {column: value}, "std": {column: value}}`
        with every column of `FEATURE_COLUMNS`.
        """
        mean = np.array(
            [feature_stats["mean"][column] for column in cls.FEATURE_COLUMNS]
        )
        std = np.array([feature_stats["std"][column] for column in cls.FEATURE_COLUMNS])
        if not np.all(std > 0):
            raise ValueError(f"Every std in feature_stats must be > 0, got {std}")
        return lambda x: (x - mean) / std

    @classmethod
    def build_dataset(
        cls,
        session: Optional[str],
        feature_stats: dict,
        controls_only: bool = True,
    ) -> NipoppyDataset:
        """Build the NipoppyDataset used for training."""
        phenotypes = [cls.SEX] + ([cls.DIAGNOSIS] if controls_only else [])
        return NipoppyDataset(
            phenotypes=phenotypes,
            target=cls.AGE,
            derivatives=None,  # set on the node by `select_freesurfer_derivatives`
            derivatives_initializer=cls.select_freesurfer_derivatives,
            session_filters=session,
            drop_na=False,  # missing values are handled in `prepare_dataframe`
            whole_df_level_transform=lambda df: cls.prepare_dataframe(
                df, controls_only
            ),
            sample_level_transform=cls.standardizer(feature_stats),
        )

    def training_data(self):
        model_args = self.model_args()
        session = model_args.get("session_by_node", {}).get(
            self.node_id(), model_args.get("default_session")
        )
        dataset = self.build_dataset(
            session,
            feature_stats=model_args["feature_stats"],
            controls_only=model_args.get("controls_only", True),
        )
        return DataManager(dataset=dataset, shuffle=True)

## Datasets on the nodes

Use the node IDs listed here to fill `session_by_node` below.

In [ ]:
from fedbiomed.researcher.requests import Requests
from fedbiomed.researcher.config import config

all_datasets = Requests(config).list()

In [ ]:
all_datasets

## Experiment

In [ ]:
model_args = {
    # fedbiomed
    "n_features": BrainAgeTrainingPlan.N_FEATURES,
    # model
    "eta0": 0.01,
    "random_state": 424242,
    "learning_rate": "invscaling",
    "penalty": "l2",
    # training plan
    "session_by_node": {"NODE_...": "01", "NODE_...": "baseline"},
    "default_session": "1",
    "feature_stats": {
        "mean": {column: 0 for column in BrainAgeTrainingPlan.FEATURE_COLUMNS},
        "std": {column: 1 for column in BrainAgeTrainingPlan.FEATURE_COLUMNS},
    },
    "controls_only": True,
}

In [ ]:
training_args = {
    "num_updates": 50,
    "loader_args": {"batch_size": 32},
}

In [ ]:
from fedbiomed.researcher.aggregators.fedavg import FedAverage
from fedbiomed.researcher.federated_workflows import Experiment

exp = Experiment(
    tags=["nipoppy"],
    training_plan_class=BrainAgeTrainingPlan,
    model_args=model_args,
    training_args=training_args,
    round_limit=10,
    aggregator=FedAverage(),
)

In [ ]:
exp.run()